# Head-memory: private FP16 state + native CE replay

**Actual prepared source; T4 UNTESTED.** 24/24 focused CPU checks passed, including
a tiny real Qwen2 fixture. This is a native-replay memory control, not full H0.
No original Triton kernels are present: official own-kernel checks 3/4 will fail.
No speed, peak-memory, quality, score or admission claim has been established.

The fixed task/harness are preserved. An authorized run measures a fresh native
baseline and candidate in the same environment, with base/solution/solution/base
order. It captures full command wall time, external nvidia-smi peaks, quality and
immediate native-route telemetry. GCP is not official Kaggle admission.
No package installation or model/data download is performed by this notebook.


In [ ]:
from pathlib import Path
import os
import sys

# Set existing prepared paths. Do not point at an output from an earlier run.
PREPARED_REPO = Path(os.environ.get("SOUP_OFFICIAL_REPO", "/absolute/prepared/pinned-official-checkout"))
PUBLIC_HF_CACHE = Path(os.environ.get("SOUP_PUBLIC_HF_CACHE", "/absolute/prepared/public-hf-hub-cache"))
OUTPUT = Path(os.environ.get("SOUP_EXPERIMENT_OUT", "/absolute/new/head-experiment-01"))
BACKEND = "GCP"  # or KAGGLE; neither reuses historical baseline timings
NATIVE_TIMEOUT_S = 900  # fixed before measuring both sides, NOT the Kaggle79s reference
OUTER_TIMEOUT_S = 8100
AUTHORIZE_GPU_EXECUTION = False  # set True only after separate current run approval


## Frozen reviewed source

The next cells materialize only this packet into a fresh temporary directory;
they do not run training. ZIP SHA256 and every solution member are verified.
The driver will separately verify pinned official source, exact direct packages,
Python3.13.15, cached public data/model revision, idle physical GPU memory and one
visible Tesla T4. Missing prerequisites stop; they do not trigger downloads.


In [ ]:
EMBEDDED_FILES = {'candidate.zip': 'UEsDBBQAAAAIAAAAIVzTmMdocAIAACMFAAAUAAAAc29sdXRpb24vX19pbml0X18ucHmVVMFq3DAQvfsrpjp56cZ0Q1tKYA8hcUho2CyJUyilCMUer9XIkpHktCHsv3dsy+vdtIdWJ1uaefPmaZ4YY2srn4THo0qoErTw8gmPzlKoUBRHNdbGPoNFh8LmFeRGe2tUAisDxsqN1EJBZqU3GlwlLCaMsai0pgZZN8Z6JR+SGr0ohBdhC9YifxQbXBl/YVpdpNYaO4cntE4aHYWgH46+I14JXSiEJRXUGHGPCgmOKIWdKCqwBNE06jmenURAa6PMA7EKqXOYkvpjWY5HIB1o43sgoJ1xPxF5J8KAdpiS5MolnEstPedj/nhWGfM4JXXLCukQblvtZY19nzG7JF07HZ23be6NBfNTU+uVbCAnoA0WIEqPFiSFCKXYbAdp0bdWk94t9nv4q8HcU8ISXhhB5RU7AXacLBbJu7d5uzj+xObAvBXalcbWVKU7/5AsPiYLtu0hSJWJMvXdknKEpkWNJ+OVxN3fDAgCui8itqu8DTxybP5+ryBcdzzVaKzUPmZ3N/drfpmenvPVaXb1JeV3n6/W6/ScCJMwMaXM5lCq1lXLrt0/NLgQyuF4n4H3m+WO1//U6yYtKdq6cfELCy13Qg2o23/l0Q99MviHd+YZxz07GL8hLNztGBJ+eSO1xqKPOxj1HUQ8EJhscZgZT1mzVygJ1pJ0CPFYUOOjgbnvDcwf0WpUbvnt+/xgisdFmb51S5a95/erLL3L0p2at+n6+vQrP7tZZbc312Fk98d18GkhxUYb52XuRreGoJdt77KpafJW70sihPttOC0aVxlPSgyY9DbR6ONrPLZnsW7qyah8an776hnYKxXMvIONfgNQSwMEFAAAAAgAAAAhXHr3Uc46BAAALg4AABMAAABzb2x1dGlvbi9pbnN0YWxsLnB5rVdbj9tEFH73rxjmydkabwsCiUWRQECh6u52oUU8IGQN9kkyWmfGnRknCqj/vWcutseXZJFKXpKx53znO/cTSukbxbdcsJqUUmij2tJIdd0o2YAyp8/lUUBFOL5hdc0MlyIn95K8U4wLUNcbqY5MVaRhptzllNJko+SebFqBMLLWhO8bqQw5KtboJBwQrYHSdMcjsEcFm8SL5gK1HKDYAas66Xv36Bd8ciu1TpKkrJnW5FXE6iYh+KlgQ4qCC26KItVQbzJS1jojAbNG6Yx0thUV6FLxBu3NiIEa9mDUaeWR7McC5E7e/YpAyHqGO5Za1OHe9IoQ4zKTKSIcuGy1s87rzztTxzf3soLaUgyOzf/A79dw+pGX1lNMndLVWIKV1gyUeKdaSPp337mopXP1g7h1eJ83qdOckSumtuicq6vHo/0VeXTZmnNyczFmDCt3/v4qGRuxk/LR+qVj07+NPTW60KdMgPUJ48EHznxDzKmBoJRwTYQ0fW4QqcgWDCIofyNH/A3fZoS6Y2FlaYY1I2BFPlsT+v4I4gu64JM+8Dns0SlBXj/ypoEKIRQwLcWauuSnY+8oMK0SMeczpA6yZH8Xmv8zIvXiqxfffPn1J3BysG2NufUksVD9eSBYYA0bXqaRj5GmrajCdRHMWDr2+kLVfAJz7GDAt6LoUJ/k7x06IjiiF/WE/4GWhblMaccq29M4JiJmVzH2HHqbHJjSvWejg41AOhUYVNniR48g6Lj9plMrp40tI3ZoqHWo0g5vyW/rTstSB/vTff2Fl9JwK4uMnbSwZb86bb660buroeIVaEwbcLaMa72Po2+KNwuuJ//SaFbSG0JZjXGrTkWAtYGk3e/CW4PXnn8YetIgb6PWRds5DlMRNKgDwsTEuo4z9DLMur7zLaTapO/N++5IZMKotySi3Ao7eJ73T5A1Ca17MUI2+WquTRoFNecG9jpdTYbC+boKwOPrQQR1zZ+fzzXHanYfScEyCmIMLTSb1dZqJuRd9Az76Zn5+pKhsv7dmWyKTotp5LR8wC3IJnLYzIqGC1zU0uka41Yqo5jQGKw9KN2FwU0hf+BiW7hjt279ag8vpfqBtZrVt3eXkKxwa/iw6T0o8MthdWffn5G1nvTxGQlHSm9dZ7l98/Ztcff9w8Or+58dlFu7wgo25ZlN5UO5d1MCRc4MH4c6HTpJyLLUFxKa29aApWSH+H91KR3lCFKIi3ICOlbzhOD7ltWC7Ts+U1f0V2cEdky7pI7xbG4XvtAqFFlNZVy0LnlgEs6Z0gAwIT2JFo3p+cFymVacHH6YRYjD0jDazoNolBTh1pnMmKTzmSyJkB2WdmUpSkjjzb5bMVZjBvlGg5Xx69jQilCtBvIbFjvfw09KSZXS34VuG1sqUF2HoeHKwJeETZOojbtp8m1nnW0TbghGrSf+B5X2pZWR5X9GyUdQSwMEFAAAAAgAAAAhXEs7ouCkCQAAlhsAABcAAABzb2x1dGlvbi9uYXRpdmVfaGVhZC5weZVZW3PbuBV+169A2YdSqUzHTZOdOsOZ5mJvMrOx0423D/VkODAJSYhJgEuQUpSd/Pd+BwApUJIVLx88IoFz+84VcBRF141cSMVLVjdyxVtxcvnx7MWJafHzVPFWrsTJmwvWiLrkm5fs6vqGccV0T3TTyFYrdi8aJcpkMrlW5YaV2hhm+EoUjg/TayUas5Q1y5dcLYRJ2M1SMMeeLQUvZpDQdmBSgHohWzOb3DRcgoxVom1kbmbsV7GWCjtfffh4+nPDi085L7HOVcFMV9elBHEhlK6gWasbBtkbLyOZXDlZpqsqbINF0nij8MpbNu/Kkr1+csMWQpPEzYxJlZddIdWCyYXSDfbNrc2NXptkcrMEB6laoVqp8Rl2F1oYpnTLat20rIWFjcj1ShDpu6dWNm82p7pr6649IZ0bYQyo2VLr+2QSRdFk3uiK5bosRU58IaKy3N7oDrKaiX/9YrTqf68Fv2/EfNK/w/Z86RjZn4lSPZd5p3KnLuOGXU4mk7zkcNaNKIW1+nzC8ECR15BXQG985QVvOVvLdgk2QE+SS+dlB4eqRQjTnZgDJia+1kIZQnutm/uErCKmhZizLJNKtlkWG1HOZ/BQI3iV3jSdmDrJ9NBakpO9hqW94fF0vA5CQ/A8uMHxxrr7MRl0EJVsvXyxgvdm7MmTuRRlYQ7rcGt3fWZ/T9nZsC7nLHIqRACAOfot+a6at1Aidptue7rP032egeZjZkhO1cbRp+vfPmbvLl69za5e3bz/70U0s5GQFF1Vm3hEQc8fkdU9Ou8tjaxJeN+3bwvD99keI4Pgye7FxlhfzRjCXa8zxVV6yUsjpjMXD86TW6iN4rVZagd3gK7LdajnNIA+hczdLq8UOHqcxqv+23RfxaivSVlra1LmahKR38K4CDWoXzFL3gh8f5o8/Y4UsJraupDlIvblh7W8WYi2jwmv8WWSNyhuGQBrdL3xm5N5qXkbT5OVFOv45Gzma1gCQbW4PTn7fEDf4PGienLY7goOkqUQX9OTs6dPqT4Wnc3dNIKy0XTI3eyjK9zveDn/1FWxy3netXqBIplc+pT3hvybKrLMkdZLXQyOQt6ueVPEefu1V35AAD/66hC4cPiW2HyKXJ3NoFrmmRFIokGEUr1MextVV4kyfhiPlc75XVeiTqa7IM5Y0W5qkVIu+TX7YbrN/BUvO0E5/6A7h60wNaEeRepmdzy/twAc3T7YDAnD7zFD0WYVfNFIXspvIiMPmNjlyJ+BzoVbNN3NGGvf5IgjB0OsJxFElVWkyLralZXAhag3BzZQY7zSSozrz8h8r3enOgNC184CXQN9idEs+Dvs2cEZgPb+KDK0VNQbM+xdysVSNJluCnT71Pc0aSy4CDJ+V4oifsBT+xD3CA3hGbJPw5ctS9v7nFwnz8qOpzsFX3dNTtHnjCN8R6qjGIm+MhRoq/kSJaMRv3cSY4CzJnYFdIwkDSmjmHaCDgQpPcRH2mLfQzWUAqu04zfz2h6tS7uPVdG526QHQudPMctB0Vog6+UDsD/Wkw9lyxaKIAL7sukGwneYPX9BRR/mnleYGBtULFaL5qTShSgZxsySnP6SdYqGNvRCDEb01TBkgB30Pr36cLGdir2T+0a4kjkGQ5pTaTTKQZ1tyxxDL8IejEyYXfPNiVRf3OjH7pW+M1SaIVe2ltdcfoVlND9zDLybeIpOvOYbQ/0cVIblXcHZKTt7fvavZy9QANDbMSGj5Jj2BDHnOc9Lvkgswyvtaya8USAf5zSfCSVogM+5wpcFh33Naa6rmlvqmR3Y+7iEdr9e/ezQSHoQH5r3vPPofBB0lZk7H6TWSfshdACztLfPY5vGERkezaa7I1wgEfkQvI23Ha/sdovVEcukJaW2e/fF0uW2H8SR0/PYLu+MpIe8nx6yb0zWBxBU6+pSxP49mLOyjPAfYO6LK6IWes1cT80MchUuQOLAw5WBZ5BBbb58CPcDMwgm/nmbObaODEMj5eAinJ1JI4d07Ma13To5Qnyc13PYQXlNY4OlTT2LQ91l18O+e6dj+9N9GNIAkUfVrAOo7X96HKsRrOHLDrzhy+M4D66Y7B0phmCl0+l+d+9je7s3HgO+F+9Ie2JlPyYtndRxADzfU9M7ykdDZLejymcCRWdnYrBFJcXxG2MNBjy7FWN7WWW0ErnqvaeVpQqUovdkLdBO2nFrHRbvJE6+DyJxQOlO4Sj9TSjSm4iRqVajaYgycZNGKgxlKpw6bQO+sQPNtAct3OdDc7Tv/FCs99qAgR+Q9hV4YDLq5fquNILlB7KyvoVmNdpAIPGvdIFDvW+tWS0V3dv8Zy0UyzWdjNAYeUkdBd1tPrfVjdFE5Mfbl54k4Hapmze8M7z85QO1Y8bzXNSta5ouSQz12Sph19SOUDQRdGZ8xdNzs4dTQcH0R+SmlQwh5e5pTERHQfdxKQu03MxeUZnoe4gmCmkb+2xiJwNHTbdNm3gVxo9VkY4h8Wraf7/Tunz0LEQNfkWXCE5eYvuxiafTPQ50pxVUhZEWWAuLCftLyqhmHw8mXmIEVjSGUfnMIB6nMzqQj0IrdtYNJXMwU6qW5G5XSOpDne6QNT4iVSErIn0WfBtOfbSwlXCEi6M49yQOIvcxFOWOnyxN2Q+wsZSU81sTdmAJz6Ak0qWfvQs4e2FlOh12NgC1F/88ZEeQo8hhyiS56HRnggx2DHdWj2DiRoUAD/9hd0diVXQ+HQ0dxzGyliEaN8gogsoRjWEKLKLZ1GaLxYFeE5R7T4URpuZ3spTtcK3iFqakfvzTjD3/cWU01U/PA/HhbXB6oIlv56x+rM/s9RM2u6APGNiodxF/LLXjoLoH1Dsl/mhxIIgCUpceFLB7Ky6y0uORtcuavBwyGfWDH1LHI/lB8ECLKK+76Af4uGefCdGP3T7d6287XjoemlsJNlkaHF5G/cv/Q4COeeIrrzBY++v9/rKfDoGNqDDaGEYX9SiMSU/15gI9weiA3RLglGhSPfUVvzp9r+b28uulP4eh0uf2uCib4WyKtqbZN9Hobf/a3olcJjVO7P2QEGMCp8tB2yDsRD69TZIEH88/JwcLQqtbGJTuXhEm7vC4d883HsxH8D8qqscOGSdf6PFWx1az3tMBjG/lStr/iPTIX/928+n92wt7yM470+oKoVDg4AGn5W2JAyh0Sojutb8OCLj5fwsZ9nv/vwnV++9vhl1+fPYPxosvmrLanfqtN0gUTXh2y9mLJGD4PzjqVImF4/LW55Nyp2b6TwjZuIC4QpTyTjTAHCoenFJ8vDoXnYb4TP4PUEsBAhQDFAAAAAgAAAAhXNOYx2hwAgAAIwUAABQAAAAAAAAAAAAAAKSBAAAAAHNvbHV0aW9uL19faW5pdF9fLnB5UEsBAhQDFAAAAAgAAAAhXHr3Uc46BAAALg4AABMAAAAAAAAAAAAAAKSBogIAAHNvbHV0aW9uL2luc3RhbGwucHlQSwECFAMUAAAACAAAACFcSzui4KQJAACWGwAAFwAAAAAAAAAAAAAApIENBwAAc29sdXRpb24vbmF0aXZlX2hlYWQucHlQSwUGAAAAAAMAAwDIAAAA5hAAAAAA', 'candidate-files.json': 'ewogICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICJhMzdlYzgxMGZlY2VlODIwODY3NDM5ZGRiNTI4MmM1YjY4YzA4MjJlMzcxY2I3NGVjZThiOGVjMGYzNDRjMWRiIiwKICAic29sdXRpb24vaW5zdGFsbC5weSI6ICJhNGRlNTljNjMzMTZkMTA3NTZlN2JkMGVlNzQ5OWQyZThjN2U2MDI4YjVhNWZiOTU0MTc5NjcwYjE3NDFmOWYxIiwKICAic29sdXRpb24vbmF0aXZlX2hlYWQucHkiOiAiYTI5MjU2NWIzMjg2MTc0Y2Y0NzQxMDcyMDZkZjNkYjQ5MmQ5Y2RkYzhlZTcwNDdmYmYyMjhlMmJkOWEyMDQ0MiIKfQo=', 'bundle-info.json': 'ewogICJzb3VyY2Vfc2hhMjU2IjogImZiZTEzM2Q0ZmRmNDYzNTMzMTJjM2RlNDk1ZjNkMzM5NThkYjA0Y2U4MDM4MGE5ODI5NjI4YmM0NGRhZjAwMjYiLAogICJmaWxlcyI6IHsKICAgICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICJhMzdlYzgxMGZlY2VlODIwODY3NDM5ZGRiNTI4MmM1YjY4YzA4MjJlMzcxY2I3NGVjZThiOGVjMGYzNDRjMWRiIiwKICAgICJzb2x1dGlvbi9pbnN0YWxsLnB5IjogImE0ZGU1OWM2MzMxNmQxMDc1NmU3YmQwZWU3NDk5ZDJlOGM3ZTYwMjhiNWE1ZmI5NTQxNzk2NzBiMTc0MWY5ZjEiLAogICAgInNvbHV0aW9uL25hdGl2ZV9oZWFkLnB5IjogImEyOTI1NjViMzI4NjE3NGNmNDc0MTA3MjA2ZGYzZGI0OTJkOWNkZGM4ZWU3MDQ3ZmJmMjI4ZTJiZDlhMjA0NDIiCiAgfSwKICAicmV2aWV3X3JlcXVpcmVkIjogdHJ1ZQp9Cg==', 'official-source-manifest.json': 'ewogICIuZ2l0aWdub3JlIjogImMzMTRiZTY0YTQxYTMyY2ZlNTVmMWFjYjVmYzYyOTM5NzEyNzZhMTQyZjQ2YWE5NjMzZmQzMjdhMmFmMmU2ZDUiLAogICJMSUNFTlNFIjogImNmYzc3NDliOTZmNjNiZDMxYzNjNDJiNWM0NzFiZjc1NjgxNDA1M2U4NDdjMTBmM2ViMDAzNDE3YmM1MjNkMzAiLAogICJOT1RJQ0UiOiAiMmYwMDg5YTg4ZTk2MjNlNWY2ZWJhNDk3Nzk2ZjI3OWMyZmRiM2ExNWJlMWRmMDk3M2U0ZDAwNTVkMzU4NzZlNCIsCiAgIlJFQURNRS5tZCI6ICJlNmE1MTE4NWZkNjEzODM4NjFmN2NhMjNlZDc3OTg5OWE3MzQxOTBkNTQ4YTJkYWE0MGIwYTdiN2EyNWM0NWY1IiwKICAiaGFybmVzcy9fdHJhaW4ucHkiOiAiNDVhNjI2NDJkY2VhYjMyY2YzZWZjZGE1MDRkZTg3YTljMjVlYTI4YjExOWIxODA2ZjU2ZDE0ZmYxMTQ2YTg5ZCIsCiAgImhhcm5lc3MvY2hlY2tfcnVsZXMucHkiOiAiODc5YWIxODYxNjc0ZjZlNzE0ZWFkZDAwYWFmMDQ0MjM1ZDk4NTI2ZGQ3NWZhYjEwMzVkZWVjMzQwMTEyMWVmNSIsCiAgImhhcm5lc3MvY29tbW9uLnB5IjogImM4MzA2NmYyNGYyNjZkYWI2MmViY2M5YzJkNDRmNWE2NTkwMjc2MGI1M2RhOTEyZTY1ZWJmZTUxMTU3OWMyZGIiLAogICJoYXJuZXNzL2V2YWxfYWRhcHRlci5weSI6ICI0N2NkZGJiOGU0NTc5YzY5Zjk1MzViMDk5ZmYyMzcxYjFjM2FmMDhiZmJkOTNkMTRiZjc5YWFkZDc0ZDM4NDgyIiwKICAiaGFybmVzcy9leGFtcGxlX3JlZmVyZW5jZS5weSI6ICI2YTExMDU1ZjRhMTgyMmU1Mzk3MWMyMjhjNmUzZTNhYTZiODhlZGQ5ZWI2ZGI4MWEzODhlMjZhNzBhMDMwY2ViIiwKICAiaGFybmVzcy9ydW5fdGFzay5weSI6ICJkOTM1YTg5NTlmMzQ4OTc3MzEzZjJlZWYzNjg3MGZmMjY5OWI5YmEwMTNhNzk3ODkxOGZmMzVlMTQ4OTY4YTc5IiwKICAiaGFybmVzcy9zY29yZS5weSI6ICI3MzFjOWI2NDRkY2M0MDY5YjBiMDY5NjE2ZjQzODA2NGQxZjdmMzc2YTIwMmZlYjE1MjAxZmQ5MTBlYzY4OGM1IiwKICAidGFzay9wcmVwYXJlX2RhdGEucHkiOiAiYTg1NDNmMTFhNTVhZmY4MGRiZGM3MTRhMGU2MjM5MTI0NGE1Mjk2NjE2ZWVhMmUzNGRjMmViM2ZmM2VjMThmNyIsCiAgInRhc2svdGFzay55YW1sIjogImRkNzBjNzM5ZmNkMTQzZTA2YWVmM2JiMWVmY2M5Mzg2ZjY4MmJmMWRlMjZmMzBlMDg0MDlkODZhOTdmZjQ3Y2UiLAogICJ0ZW1wbGF0ZXMvU1VCTUlTU0lPTi5tZCI6ICJlNTAxNGUzZmYwZTMxZGE3YTdjYzNhMmVhZjk4OGFlZDExMjU2ZjhmMTAwZmNjNTUzNjM1OTA1YmFkOTEzNjY4IiwKICAidGVtcGxhdGVzL2thZ2dsZV9ub3RlYm9vay5pcHluYiI6ICI1OGM5NDE1MDRlY2RiNDllOWVlMTA5MTg5MDVlYjBiNjYxNWUyNzcyNWZlYWIwMTVjMDFlOTExNTBlZGY3OTcyIiwKICAidGVzdHMvY29uZnRlc3QucHkiOiAiYTgyZWFiYWI0NjhlYTBlZjFlMDAzY2NjZDdjMzgyN2NlMmY0ZDAwZDA5YzMyOTQ4YTM0ZmI0ODIyMWY3MzM3YSIsCiAgInRlc3RzL3Rlc3RfY2FsbGJhY2sucHkiOiAiMTkxOTMwM2EzNjcwYzBhMzNhYmJmY2U5YjMzZWFiMDkxM2JiNjRhM2EyMWMzMzEyZDg2NTMxYzMyYjViZjQ5MCIsCiAgInRlc3RzL3Rlc3RfY29uZmlnLnB5IjogImRiODA4YzQ0NDcwZDE0NGI3Y2YxOGI3MzI5NDkyZjQxZjc2ZTM3Y2ZjMjQzMDZlN2NiYTA0ODhiY2E5ZDc4N2MiLAogICJ0ZXN0cy90ZXN0X2Zvcm1hdHMucHkiOiAiNWQ2NTZlZDUwYzU5Zjc5NGU5MzQ0NzNkNDU5ZWE4NTYzNWY4OTJiYTAyOGUxOTI0MTQ0NzRlOWI1ZDY1OTE5ZSIsCiAgInRlc3RzL3Rlc3RfbG9hZGVyLnB5IjogIjNkYWQxYmVhMmQzM2FhYjk3ZTUyYjdjZDQ0ODg4ZDc2OGJjOTUxOWFmNDRmYmU2Y2YzZTk2YzI2MDAyNjYzMmYiLAogICJ0ZXN0cy90ZXN0X3RyYWluZXJfaW5pdC5weSI6ICJiNzYwOGQ2MmE0MDk4M2JmN2M5ZTViMjcwZTBlNTI3YjY0MWE4MzdlNDYyZDZkZjI5MDlmZWM4OGY2MDdlODIxIgp9Cg==', 'run_candidate.py': '...', 'process_guard.py': 'IiIiSXNvbGF0ZWQgTGludXggc3VicmVhcGVyIGd1YXJkaWFuLiBPd25zIGNoaWxkcmVuIGFuZCBob2xkcyB0aGUgR1BVIGxvY2suCgpUaGUgZ3VhcmRpYW4gd2F0Y2hlcyBhIHBhcmVudC1saXZlbmVzcyBwaXBlOiBkcml2ZXIgU0lHS0lMTCBvciBvdGhlciBhYnJ1cHQKZGVhdGggY2xvc2VzIHRoZSBwaXBlIGFuZCB0cmlnZ2VycyBxdWFyYW50aW5lZCBjbGVhbnVwLiBQSURGRCBzaWduYWxpbmcgaW4gdGhlCnNoYXJlZCB0YXNrIGhlbHBlciBwcmV2ZW50cyBQSUQgcmV1c2UgcmFjZXM7IGRlc2NlbmRhbnRzIGFyZSBvbmx5IHRoaXMgZGVkaWNhdGVkCmd1YXJkaWFuJ3MgY2hpbGRyZW4sIGluY2x1ZGluZyBuZXdseSBhZG9wdGVkIG9ycGhhbnMgZnJvbSBuZXN0ZWQgc2Vzc2lvbnMuClRoaXMgaXMgdGFzayBvcmNoZXN0cmF0aW9uLCBuZXZlciBzb2x1dGlvbi8gd29yayBvciBrZXJuZWwgcGFydGljaXBhdGlvbi4KIiIiCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNlbGVjdAppbXBvcnQgc2lnbmFsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKCmltcG9ydCBvd25lZF9wcm9jZXNzX3RyZWUgYXMgdHJlZQoKCmRlZiBxdWFyYW50aW5lKHBhdGgsIHJlYXNvbik6CiAgICAiIiJGaXJzdCBmYWlsdXJlIHdpbnM7IG5ldmVyIG92ZXJ3cml0ZSBvciBhdXRvbWF0aWNhbGx5IGNsZWFyIGEgbWFya2VyLiIiIgogICAgdHJ5OgogICAgICAgIGRlc2NyaXB0b3IgPSBvcy5vcGVuKHBhdGgsIG9zLk9fV1JPTkxZIHwgb3MuT19DUkVBVCB8IG9zLk9fRVhDTCB8IG9zLk9fTk9GT0xMT1csIDBvNjAwKQogICAgZXhjZXB0IEZpbGVFeGlzdHNFcnJvcjoKICAgICAgICByZXR1cm4KICAgIHdpdGggb3MuZmRvcGVuKGRlc2NyaXB0b3IsICJ3IikgYXMgc3RyZWFtOgogICAgICAgIGpzb24uZHVtcCh7InN0YXR1cyI6ICJPUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCIsICJyZWFzb24iOiByZWFzb24sCiAgICAgICAgICAgICAgICAgICAiZ3VhcmRpYW5fcGlkIjogb3MuZ2V0cGlkKCksICJjcmVhdGVkX3VuaXgiOiB0aW1lLnRpbWUoKX0sIHN0cmVhbSkKICAgICAgICBzdHJlYW0ud3JpdGUoIlxuIikKICAgICAgICBzdHJlYW0uZmx1c2goKQogICAgICAgIG9zLmZzeW5jKHN0cmVhbS5maWxlbm8oKSkKCgpkZWYgX3dyaXRlX3JlY2VpcHQocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBlbmRpbmcgPSBwYXRoLndpdGhfbmFtZShwYXRoLm5hbWUgKyAiLnBlbmRpbmciKQogICAgcGVuZGluZy53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUsIGluZGVudD0yKSArICJcbiIpCiAgICBwZW5kaW5nLnJlcGxhY2UocGF0aCkKCgpkZWYgZ3VhcmRpYW5fbWFpbigpOgogICAgY29uZmlnID0ganNvbi5sb2FkcyhzeXMuc3RkaW4ucmVhZGxpbmUoKSkKICAgIHJlYWRfZmQsIGxvY2tfZmQgPSBjb25maWdbInBhcmVudF9waXBlX2ZkIl0sIGNvbmZpZ1sibG9ja19mZCJdCiAgICBzdGFydGVkLCByZXF1ZXN0ZWRfc2lnbmFsLCBwcm9jZXNzID0gdGltZS5tb25vdG9uaWMoKSwgW05vbmVdLCBOb25lCiAgICByZXN1bHQgPSB7ImV4aXRfY29kZSI6IDEyNSwgInJvb3RfZXhpdF9jb2RlIjogTm9uZSwgInRpbWVkX291dCI6IEZhbHNlLAogICAgICAgICAgICAgICJpbnRlcnJ1cHRlZCI6IEZhbHNlLCAiY2xlYW51cF9jb25maXJtZWQiOiBGYWxzZSwKICAgICAgICAgICAgICAicmVhc29uIjogTm9uZSwgImd1YXJkaWFuX3BpZCI6IG9zLmdldHBpZCgpfQogICAgZGVmIHJlcXVlc3Qoc2lnbnVtLCBmcmFtZSk6CiAgICAgICAgcmVxdWVzdGVkX3NpZ25hbFswXSA9IHNpZ251bQogICAgZm9yIHNpZyBpbiAoc2lnbmFsLlNJR0lOVCwgc2lnbmFsLlNJR1RFUk0pOgogICAgICAgIHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KQogICAgdHJ5OgogICAgICAgIHRyZWUucHJlcGFyZSgpICAjIHJlcXVpcmVkIGJlZm9yZSBhbnkgY2hpbGQ7IG5ldmVyIHNpbGVudGx5IGRlZ3JhZGUKICAgICAgICB3aXRoIFBhdGgoY29uZmlnWyJsb2ciXSkub3BlbigieCIpIGFzIHN0cmVhbToKICAgICAgICAgICAgcHJvY2VzcyA9IHN1YnByb2Nlc3MuUG9wZW4oY29uZmlnWyJjb21tYW5kIl0sIGN3ZD1jb25maWdbImN3ZCJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdGRvdXQ9c3RyZWFtLCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0YXJ0X25ld19zZXNzaW9uPVRydWUpCiAgICAgICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgICAgICByZWFkeSwgXywgXyA9IHNlbGVjdC5zZWxlY3QoW3JlYWRfZmRdLCBbXSwgW10sIDAuMDUpCiAgICAgICAgICAgICAgICBpZiByZWFkeSBhbmQgb3MucmVhZChyZWFkX2ZkLCAxKSA9PSBiIiI6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249InBhcmVudF9saXZlbmVzc19waXBlX2Nsb3NlZCIsIGludGVycnVwdGVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGlmIHJlcXVlc3RlZF9zaWduYWxbMF0gaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX3NpZ25hbF8iICsgc3RyKHJlcXVlc3RlZF9zaWduYWxbMF0pLCBpbnRlcnJ1cHRlZD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBpZiB0aW1lLm1vbm90b25pYygpIC0gc3RhcnRlZCA+PSBjb25maWdbInRpbWVvdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHQudXBkYXRlKHJlYXNvbj0ib3V0ZXJfdGltZW91dCIsIHRpbWVkX291dD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICByZXR1cm5jb2RlID0gcHJvY2Vzcy5wb2xsKCkKICAgICAgICAgICAgICAgIGlmIHJldHVybmNvZGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyb290X2V4aXRfY29kZSJdID0gcmV0dXJuY29kZQogICAgICAgICAgICAgICAgICAgIGlmIHRyZWUuZGVzY2VuZGFudHMoKToKICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyZWFzb24iXSA9ICJyb290X2V4aXRlZF93aXRoX293bmVkX2Rlc2NlbmRhbnRzIgogICAgICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUoZXhpdF9jb2RlPXJldHVybmNvZGUsIGNsZWFudXBfY29uZmlybWVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgaWYgcmVzdWx0WyJyZWFzb24iXSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHF1YXJhbnRpbmUoY29uZmlnWyJtYXJrZXIiXSwgcmVzdWx0WyJyZWFzb24iXSkKICAgICAgICAgICAgICAgICMgY2xlYW51cCBpZ25vcmVzIHJlcGVhdGVkIElOVC9URVJNLCBzdG9wcy9yZXNjYW5zL2tpbGxzIGJ5CiAgICAgICAgICAgICAgICAjIHBpZGZkIGFuZCB3YWl0cy9yZWFwcyB3aXRob3V0IGFueSB0aW1lb3V0LXVubG9jayBlc2NhcGUuCiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUocm9vdF9leGl0X2NvZGU9cHJvY2Vzcy5yZXR1cm5jb2RlLCBjbGVhbnVwX2NvbmZpcm1lZD1UcnVlKQogICAgICAgICAgICAgICAgaWYgcmVzdWx0WyJ0aW1lZF9vdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTI0CiAgICAgICAgICAgICAgICBlbGlmIHJlc3VsdFsiaW50ZXJydXB0ZWQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTMwCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX2V4Y2VwdGlvbjoiICsgcmVwcihleGMpLCBleGl0X2NvZGU9MTI1KQogICAgICAgIHRyeToKICAgICAgICAgICAgcXVhcmFudGluZShjb25maWdbIm1hcmtlciJdLCByZXN1bHRbInJlYXNvbiJdKQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIGlmIHByb2Nlc3MgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgcmVzdWx0WyJjbGVhbnVwX2NvbmZpcm1lZCJdID0gbm90IHRyZWUuZGVzY2VuZGFudHMoKQogICAgZmluYWxseToKICAgICAgICByZXN1bHRbIm91dGVyX3dhbGxfcyJdID0gdGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0ZWQKICAgICAgICBfd3JpdGVfcmVjZWlwdChjb25maWdbInJlY2VpcHQiXSwgcmVzdWx0KQogICAgICAgIG9zLmNsb3NlKHJlYWRfZmQpCiAgICAgICAgIyBUaGUgaW5oZXJpdGVkIG9wZW4tZmlsZSBkZXNjcmlwdGlvbiByZXRhaW5zIHRoZSBjYW5vbmljYWwgZmxvY2sgZXZlbgogICAgICAgICMgaWYgdGhlIGRyaXZlciBoYXMgZGllZC4gQ2xvc2UgT05MWSBhZnRlciBvd25lZC10cmVlIGNsZWFudXAgY29tcGxldGVzLgogICAgICAgIG9zLmNsb3NlKGxvY2tfZmQpCiAgICByZXR1cm4gcmVzdWx0WyJleGl0X2NvZGUiXQoKCmRlZiBydW4oY29tbWFuZCwgKiwgY3dkLCBlbnYsIGxvZywgdGltZW91dCwgbG9ja19mZCwKICAgICAgICBtYXJrZXI9Ii90bXAvc291cC1ncHUwLmxvY2suYmxvY2tlZCIpOgogICAgIiIiRGVkaWNhdGVkIGRyaXZlciBBUEkuIEFueSBCYXNlRXhjZXB0aW9uIGFza3MgZ3VhcmRpYW4gY2xlYW51cCBhbmQgd2FpdHMuIiIiCiAgICByZWNlaXB0ID0gUGF0aChzdHIobG9nKSArICIucHJvY2Vzcy5qc29uIikucmVzb2x2ZSgpCiAgICBpZiByZWNlaXB0LmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiUHJvY2VzcyByZWNlaXB0IGFscmVhZHkgZXhpc3RzOyByZWZ1c2UgdG8gb3ZlcndyaXRlIikKICAgIHJlYWRfZmQsIHdyaXRlX2ZkID0gb3MucGlwZSgpCiAgICBjb25maWcgPSB7ImNvbW1hbmQiOiBbc3RyKHgpIGZvciB4IGluIGNvbW1hbmRdLCAiY3dkIjogc3RyKFBhdGgoY3dkKS5yZXNvbHZlKCkpLAogICAgICAgICAgICAgICJsb2ciOiBzdHIoUGF0aChsb2cpLnJlc29sdmUoKSksICJ0aW1lb3V0IjogdGltZW91dCwKICAgICAgICAgICAgICAicmVjZWlwdCI6IHN0cihyZWNlaXB0KSwgInBhcmVudF9waXBlX2ZkIjogcmVhZF9mZCwKICAgICAgICAgICAgICAibG9ja19mZCI6IGxvY2tfZmQsICJtYXJrZXIiOiBzdHIobWFya2VyKX0KICAgIGd1YXJkaWFuID0gTm9uZQogICAgaW50ZXJydXB0ZWQgPSBOb25lCiAgICBkZWYgcmVxdWVzdChzaWdudW0sIGZyYW1lKToKICAgICAgICByYWlzZSBJbnRlcnJ1cHRlZEVycm9yKCJEcml2ZXIgcmVjZWl2ZWQgc2lnbmFsICIgKyBzdHIoc2lnbnVtKSkKICAgIG9sZF9oYW5kbGVycyA9IHtzaWc6IHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KSBmb3Igc2lnIGluIChzaWduYWwuU0lHSU5ULCBzaWduYWwuU0lHVEVSTSl9CiAgICB0cnk6CiAgICAgICAgZ3VhcmRpYW4gPSBzdWJwcm9jZXNzLlBvcGVuKFtzeXMuZXhlY3V0YWJsZSwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKSksICItLWd1YXJkaWFuIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0ZGluPXN1YnByb2Nlc3MuUElQRSwgZW52PWVudiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcGFzc19mZHM9KHJlYWRfZmQsIGxvY2tfZmQpLCBzdGFydF9uZXdfc2Vzc2lvbj1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0ZXh0PVRydWUpCiAgICAgICAgb3MuY2xvc2UocmVhZF9mZCkKICAgICAgICByZWFkX2ZkID0gTm9uZQogICAgICAgIGd1YXJkaWFuLnN0ZGluLndyaXRlKGpzb24uZHVtcHMoY29uZmlnKSArICJcbiIpCiAgICAgICAgZ3VhcmRpYW4uc3RkaW4uY2xvc2UoKQogICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIGludGVycnVwdGVkID0gcmVwcihleGMpCiAgICAgICAgIyBJZ25vcmUgcmVwZWF0ZWQgc3RvcCBzaWduYWxzIHVudGlsIHRoZSBndWFyZGlhbiBjb25maXJtcyBjbGVhbnVwLgogICAgICAgIGZvciBzaWcgaW4gb2xkX2hhbmRsZXJzOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgc2lnbmFsLlNJR19JR04pCiAgICAgICAgaWYgd3JpdGVfZmQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIG9zLmNsb3NlKHdyaXRlX2ZkKQogICAgICAgICAgICB3cml0ZV9mZCA9IE5vbmUKICAgICAgICBpZiBndWFyZGlhbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgaWYgZ3VhcmRpYW4uc3RkaW4gaXMgbm90IE5vbmUgYW5kIG5vdCBndWFyZGlhbi5zdGRpbi5jbG9zZWQ6CiAgICAgICAgICAgICAgICBndWFyZGlhbi5zdGRpbi5jbG9zZSgpCiAgICAgICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZmluYWxseToKICAgICAgICBpZiByZWFkX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZShyZWFkX2ZkKQogICAgICAgIGlmIHdyaXRlX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZSh3cml0ZV9mZCkKICAgICAgICBmb3Igc2lnLCBoYW5kbGVyIGluIG9sZF9oYW5kbGVycy5pdGVtcygpOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgaGFuZGxlcikKICAgIGlmIG5vdCByZWNlaXB0LmlzX2ZpbGUoKToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX21pc3NpbmdfcmVjZWlwdCIpCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJHdWFyZGlhbiByZWNlaXB0IG1pc3Npbmc7IEdQVSBxdWFyYW50aW5lIHJlcXVpcmVzIG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uIikKICAgIHJlc3VsdCA9IGpzb24ubG9hZHMocmVjZWlwdC5yZWFkX3RleHQoKSkKICAgIGlmIGludGVycnVwdGVkIGlzIG5vdCBOb25lOgogICAgICAgIHJlc3VsdC51cGRhdGUocGFyZW50X2V4Y2VwdGlvbj1pbnRlcnJ1cHRlZCwgaW50ZXJydXB0ZWQ9VHJ1ZSkKICAgICAgICBpZiByZXN1bHRbImV4aXRfY29kZSJdID09IDA6CiAgICAgICAgICAgIHJlc3VsdFsiZXhpdF9jb2RlIl0gPSAxMzAKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImRyaXZlcl9pbnRlcnJ1cHRpb246IiArIGludGVycnVwdGVkKQogICAgaWYgbm90IHJlc3VsdFsiY2xlYW51cF9jb25maXJtZWQiXToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX3VuY2VydGFpbl9jbGVhbnVwIikKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkd1YXJkaWFuIGNsZWFudXAgdW5jZXJ0YWluOyBxdWFyYW50aW5lIHJldGFpbmVkIikKICAgIHJldHVybiByZXN1bHQKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaWYgc3lzLmFyZ3ZbMTpdICE9IFsiLS1ndWFyZGlhbiJdOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIkludGVybmFsIGd1YXJkaWFuIGVudHJ5cG9pbnQgb25seSIpCiAgICByYWlzZSBTeXN0ZW1FeGl0KGd1YXJkaWFuX21haW4oKSkK', 'owned_process_tree.py': 'IyBTaGFyZWQgdGFzayBvcmNoZXN0cmF0aW9uIGZyb20gdGhlIExvUkEgbGFuZTsgbm8ga2VybmVsIG93bmVyc2hpcCBjcmVkaXQuCiIiIkxpbnV4LW9ubHkgZGVkaWNhdGVkLXJ1bm5lciBjbGVhbnVwOyBubyBHUFUgaW1wb3J0cyBvciBiYWNrZ3JvdW5kIGFjdGl2aXR5LgoKT3JpZ2luYWwgdGFzay1vcmNoZXN0cmF0aW9uIGhlbHBlci4gTm90IHBhcnQgb2Ygc29sdXRpb24vIG9yIG93bi1rZXJuZWwgY3JlZGl0LgoiIiIKaW1wb3J0IGN0eXBlcwppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaWduYWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQoKCmRlZiBwcmVwYXJlKCk6CiAgICAiIiJGYWlsIGJlZm9yZSBHUFUgd29yayBpZiByYWNlLXNhZmUgcHJvY2VzcyBvd25lcnNoaXAgaXMgdW5hdmFpbGFibGUuIiIiCiAgICBpZiBub3QgaGFzYXR0cihvcywncGlkZmRfb3BlbicpIG9yIG5vdCBoYXNhdHRyKHNpZ25hbCwncGlkZmRfc2VuZF9zaWduYWwnKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0xpbnV4IHBpZGZkIHByb2Nlc3Mgb3duZXJzaGlwIHJlcXVpcmVkJykKICAgIGZkPW9zLnBpZGZkX29wZW4ob3MuZ2V0cGlkKCkpO29zLmNsb3NlKGZkKQogICAgbGliYz1jdHlwZXMuQ0RMTChOb25lLHVzZV9lcnJubz1UcnVlKQogICAgaWYgbGliYy5wcmN0bCgzNiwxLDAsMCwwKSE9MDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0Nhbm5vdCBlbmFibGUgY2hpbGQgc3VicmVhcGVyOyBubyBHUFUgam9iIHN0YXJ0ZWQnKQogICAgZW5hYmxlZD1jdHlwZXMuY19pbnQoMCkKICAgIGlmIGxpYmMucHJjdGwoMzcsY3R5cGVzLmJ5cmVmKGVuYWJsZWQpLDAsMCwwKSE9MCBvciBlbmFibGVkLnZhbHVlIT0xOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignQ2hpbGQgc3VicmVhcGVyIHZlcmlmaWNhdGlvbiBmYWlsZWQnKQoKCmRlZiBpZGVudGl0eShwaWQpOgogICAgdHJ5OgogICAgICAgIGZpZWxkcz1QYXRoKGYnL3Byb2Mve3BpZH0vc3RhdCcpLnJlYWRfdGV4dCgpLnJzcGxpdCgnKScsMSlbMV0uc3BsaXQoKQogICAgICAgIHJldHVybiBpbnQoZmllbGRzWzFdKSxpbnQoZmllbGRzWzE5XSksZmllbGRzWzBdCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcixJbmRleEVycm9yKTpyZXR1cm4gTm9uZQoKCmRlZiBkZXNjZW5kYW50cygpOgogICAgcmVjb3Jkcz17fQogICAgZm9yIHAgaW4gUGF0aCgnL3Byb2MnKS5pdGVyZGlyKCk6CiAgICAgICAgaWYgcC5uYW1lLmlzZGlnaXQoKToKICAgICAgICAgICAgcmVjPWlkZW50aXR5KGludChwLm5hbWUpKQogICAgICAgICAgICBpZiByZWMgaXMgbm90IE5vbmU6cmVjb3Jkc1tpbnQocC5uYW1lKV09cmVjCiAgICBvd25lZD17b3MuZ2V0cGlkKCl9CiAgICB3aGlsZSBUcnVlOgogICAgICAgIGFkZD17cGlkIGZvciBwaWQsKHBhcmVudCxfLF8pIGluIHJlY29yZHMuaXRlbXMoKSBpZiBwYXJlbnQgaW4gb3duZWR9LW93bmVkCiAgICAgICAgaWYgbm90IGFkZDpicmVhawogICAgICAgIG93bmVkLnVwZGF0ZShhZGQpCiAgICByZXR1cm4ge3BpZDpyZWNvcmRzW3BpZF0gZm9yIHBpZCBpbiBvd25lZCBpZiBwaWQhPW9zLmdldHBpZCgpIGFuZCBwaWQgaW4gcmVjb3Jkc30KCgpkZWYgc2lnbmFsX2V4YWN0KHBpZCxiaXJ0aCxzaWcpOgogICAgIiIiTmV2ZXIgc2lnbmFsIGEgcmVjeWNsZWQgUElELiBUaGUgRkQgcGlucyBpZGVudGl0eSB0aHJvdWdoIHRoZSBzZW5kLiIiIgogICAgdHJ5OmZkPW9zLnBpZGZkX29wZW4ocGlkKQogICAgZXhjZXB0IFByb2Nlc3NMb29rdXBFcnJvcjpyZXR1cm4KICAgIHRyeToKICAgICAgICBub3c9aWRlbnRpdHkocGlkKQogICAgICAgIGlmIG5vdyBpcyBub3QgTm9uZSBhbmQgbm93WzFdPT1iaXJ0aDoKICAgICAgICAgICAgdHJ5OnNpZ25hbC5waWRmZF9zZW5kX3NpZ25hbChmZCxzaWcsTm9uZSwwKQogICAgICAgICAgICBleGNlcHQgUHJvY2Vzc0xvb2t1cEVycm9yOnBhc3MKICAgIGZpbmFsbHk6b3MuY2xvc2UoZmQpCgoKZGVmIF90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWcpOgogICAgdHJ5OnNpZ25hbF9leGFjdChwaWQsYmlydGgsc2lnKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgICMgRG8gbm90IHVud2luZC9yZWxlYXNlIEdQVSBvd25lcnNoaXAgd2hlbiBjbGVhbnVwIGlzIHVuY2VydGFpbi4KICAgICAgICBxdWFyYW50aW5lKCdDbGVhbnVwIHNpZ25hbCBmYWlsZWQ6ICcrcmVwcihleGMpKQogICAgICAgIHByaW50KCdTT1VQX0NMRUFOVVBfUEVORElORyAnK3JlcHIoZXhjKSxmaWxlPXN5cy5zdGRlcnIsZmx1c2g9VHJ1ZSkKCgpkZWYgY2xlYW51cChwcm9jKToKICAgIHNhdmVkPXtzOnNpZ25hbC5zaWduYWwocyxzaWduYWwuU0lHX0lHTikgZm9yIHMgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pfQogICAgdHJ5OgogICAgICAgIGZyb3plbj1zZXQoKQogICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgIG9ic2VydmVkPWRlc2NlbmRhbnRzKCkKICAgICAgICAgICAgbmV3PXsocGlkLHZbMV0pIGZvciBwaWQsdiBpbiBvYnNlcnZlZC5pdGVtcygpfS1mcm96ZW4KICAgICAgICAgICAgaWYgbm90IG5ldzpicmVhawogICAgICAgICAgICBmb3IgcGlkLGJpcnRoIGluIG5ldzpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR1NUT1ApCiAgICAgICAgICAgIGZyb3plbi51cGRhdGUobmV3KQogICAgICAgIGZvciBwaWQsYmlydGggaW4gZnJvemVuOl90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWduYWwuU0lHS0lMTCkKICAgICAgICBwcm9jLndhaXQoKQogICAgICAgICMgTm8gdGltZW91dC1iYXNlZCByZWxlYXNlOiBpZiBhIEdQVSBjaGlsZCBpcyBzdHVjayBpbiBrZXJuZWwgZXhpdCwKICAgICAgICAjIHJldGFpbiB0aGlzIHN1cGVydmlzb3IgYW5kIGl0cyBsb2NrIHVudGlsIGV4aXQgaXMgYWN0dWFsbHkgb2JzZXJ2ZWQuCiAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBwaWQsXz1vcy53YWl0cGlkKC0xLG9zLldOT0hBTkcpCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IHBpZDpicmVhawogICAgICAgICAgICAgICAgZXhjZXB0IENoaWxkUHJvY2Vzc0Vycm9yOmJyZWFrCiAgICAgICAgICAgIHJlbWFpbmluZz1kZXNjZW5kYW50cygpCiAgICAgICAgICAgIGlmIG5vdCByZW1haW5pbmc6YnJlYWsKICAgICAgICAgICAgZm9yIHBpZCwoXyxiaXJ0aCxfKSBpbiByZW1haW5pbmcuaXRlbXMoKTpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR0tJTEwpCiAgICAgICAgICAgIHRpbWUuc2xlZXAoLjA1KQogICAgZmluYWxseToKICAgICAgICBmb3Igc2lnLGhhbmRsZXIgaW4gc2F2ZWQuaXRlbXMoKTpzaWduYWwuc2lnbmFsKHNpZyxoYW5kbGVyKQoKCmRlZiBydW4oYXJndiwqLGN3ZCxlbnYsc3Rkb3V0LHRpbWVvdXQ9Tm9uZSxzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQpOgogICAgIiIiUnVuIGluIGRlZGljYXRlZCBzdXBlcnZpc29yIG9ubHk7IGNsZWFudXAgbmVzdGVkIHNlc3Npb25zIG9uIGFueSBmYWlsdXJlLiIiIgogICAgcHJvYz1zdWJwcm9jZXNzLlBvcGVuKGFyZ3YsY3dkPWN3ZCxlbnY9ZW52LHN0ZG91dD1zdGRvdXQsc3RkZXJyPXN0ZGVycixzdGFydF9uZXdfc2Vzc2lvbj1UcnVlKQogICAgdHJ5OgogICAgICAgIGNvZGU9cHJvYy53YWl0KHRpbWVvdXQ9dGltZW91dCkKICAgICAgICBpZiBkZXNjZW5kYW50cygpOgogICAgICAgICAgICBxdWFyYW50aW5lKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50cycpCiAgICAgICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50czsgY2xlYW5lZCBhbmQgcmVqZWN0ZWQnKQogICAgICAgIHJldHVybiBjb2RlCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcXVhcmFudGluZSgnT3V0ZXIgdGltZW91dC9pbnRlcnJ1cHRpb24vZXJyb3I6ICcrcmVwcihleGMpKQogICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICByYWlzZQoKCmRlZiBpbnN0YWxsX3NpZ25hbF9oYW5kbGVycygpOgogICAgZGVmIGludGVycnVwdGVkKHNpZ251bSxmcmFtZSk6CiAgICAgICAgcXVhcmFudGluZSgnUnVubmVyIGludGVycnVwdGVkIGJ5IHNpZ25hbCAnK3N0cihzaWdudW0pKQogICAgICAgIHJhaXNlIEludGVycnVwdGVkRXJyb3IoJ1J1bm5lciBpbnRlcnJ1cHRlZCBieSBzaWduYWwgJytzdHIoc2lnbnVtKSkKICAgIGZvciBzaWcgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pOnNpZ25hbC5zaWduYWwoc2lnLGludGVycnVwdGVkKQoKIyBBIGNyYXNoLXN1cnZpdmluZyBtYXJrZXIgaXMgcGxhY2VkIGJlZm9yZSBhbnkgR1BVIHN1YnByb2Nlc3MuIE9ubHkgaXRzIGV4YWN0CiMgb3duZXIgbWF5IGNsZWFyIGl0IG9uIGEgY2xlYW4sIG5vbi1pbnRlcnJ1cHRlZCBleGl0IHdpdGggbm8gbGl2ZSBkZXNjZW5kYW50cy4KIyBJbnRlcnJ1cHRlZC90aW1lZC1vdXQgd29yayBsZWF2ZXMgaXQgZm9yIGV4cGxpY2l0IG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uLgpfTUFSS0VSID0gTm9uZQpfVE9LRU4gPSBOb25lCl9LRUVQX01BUktFUiA9IEZhbHNlCgoKZGVmIGNoZWNrX3F1YXJhbnRpbmUocGF0aCk6CiAgICBpZiBQYXRoKHBhdGgpLmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignR1BVMCBxdWFyYW50aW5lZDogcmVjb25jaWxlICcrc3RyKHBhdGgpKycgYmVmb3JlIGFub3RoZXIgcnVuJykKCgpkZWYgYmVnaW5fcXVhcmFudGluZShwYXRoKToKICAgIGdsb2JhbCBfTUFSS0VSLF9UT0tFTixfS0VFUF9NQVJLRVIKICAgIGltcG9ydCBqc29uLHV1aWQKICAgIF9NQVJLRVI9UGF0aChwYXRoKTtfVE9LRU49dXVpZC51dWlkNCgpLmhleDtfS0VFUF9NQVJLRVI9RmFsc2UKICAgIGZkPW9zLm9wZW4oc3RyKF9NQVJLRVIpLG9zLk9fV1JPTkxZfG9zLk9fQ1JFQVR8b3MuT19FWENMfG9zLk9fTk9GT0xMT1csMG82MDApCiAgICB3aXRoIG9zLmZkb3BlbihmZCwndycpIGFzIG91dDoKICAgICAgICBqc29uLmR1bXAoeydvd25lcl90b2tlbic6X1RPS0VOLCdvd25lcl9waWQnOm9zLmdldHBpZCgpLCdzdGF0dXMnOidPV05FRF9HUFVfV09SS19JTl9QUk9HUkVTUyd9LG91dCkKICAgICAgICBvdXQuZmx1c2goKTtvcy5mc3luYyhvdXQuZmlsZW5vKCkpCgoKZGVmIHF1YXJhbnRpbmUocmVhc29uKToKICAgIGdsb2JhbCBfS0VFUF9NQVJLRVIKICAgIF9LRUVQX01BUktFUj1UcnVlCiAgICBpZiBfTUFSS0VSIGlzIE5vbmU6cmV0dXJuCiAgICBpbXBvcnQganNvbgogICAgdHJ5OnZhbHVlPWpzb24ubG9hZHMoX01BUktFUi5yZWFkX3RleHQoKSkKICAgIGV4Y2VwdCAoT1NFcnJvcixWYWx1ZUVycm9yKTpyZXR1cm4KICAgIGlmIHZhbHVlLmdldCgnb3duZXJfdG9rZW4nKSE9X1RPS0VOOnJldHVybgogICAgdmFsdWUudXBkYXRlKHN0YXR1cz0nQkxPQ0tFRF9PUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCcscmVhc29uPXN0cihyZWFzb24pKQogICAgX01BUktFUi53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUpKydcbicpCgoKZGVmIGZpbmlzaF9xdWFyYW50aW5lKCk6CiAgICBpZiBfTUFSS0VSIGlzIE5vbmUgb3IgX0tFRVBfTUFSS0VSOnJldHVybgogICAgaWYgZGVzY2VuZGFudHMoKToKICAgICAgICBxdWFyYW50aW5lKCdMaXZlIG93bmVkIGRlc2NlbmRhbnRzIHJlbWFpbiBhdCBzdXBlcnZpc29yIGV4aXQnKQogICAgICAgIHJldHVybgogICAgaW1wb3J0IGpzb24KICAgIHRyeTp2YWx1ZT1qc29uLmxvYWRzKF9NQVJLRVIucmVhZF90ZXh0KCkpCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcik6cmV0dXJuCiAgICBpZiB2YWx1ZS5nZXQoJ293bmVyX3Rva2VuJyk9PV9UT0tFTjpfTUFSS0VSLnVubGluaygpCg==', 'README.md': 'IyBIZWFkLW1lbW9yeSBjYW5kaWRhdGU6IHByaXZhdGUgRlAxNiBzdGF0ZSwgbmF0aXZlIENFIHJlcGxheQoKKipQUkVQQVJFRCBSRVNFQVJDSCBDT05UUk9MIOKAlCBUNCBVTlRFU1RFRCDigJQgTk9UIEEgQ09OVEVTVC1BRE1JU1NJQkxFIFNUQU5EQUxPTkUgU09MVVRJT04uKioKClRoaXMgaXMgYWN0dWFsIHJ1bm5hYmxlIG9wdGltaXphdGlvbiBzb3VyY2UsIG5vdCBhIHBsYWNlaG9sZGVyLiBJdCBjaGFuZ2VzIHRoZQpsb3NzIGdyYXBoJ3MgcmV0YWluZWQgc3RhdGUgd2hpbGUgZGVsaWJlcmF0ZWx5IGtlZXBpbmcgdGhlIG5hdGl2ZSBoZWFkIGFuZCBldmVyeQpyZXR1cm5lZCBsb2dpdC4gSXQgaGFzICoqemVybyBvcmlnaW5hbCBUcml0b24ga2VybmVscyoqLCBzbyBvZmZpY2lhbCBjaGVja3MgMy80CmNhbm5vdCBwYXNzLiBObyBzcGVlZCwgcGVhay1tZW1vcnksIHF1YWxpdHksIHNjb3JlLCBvciBhZG1pc3Npb24gY2xhaW0gaXMgbWFkZS4KVGhlIGZ1bGwgSDAvb3V0cHV0LXN1cHByZXNzaW9uL3N1bW1hcnktb25seSB2YXJpYW50IGlzICoqQkxPQ0tFRCoqLCBub3Qgc2lsZW50bHkKcmVwcmVzZW50ZWQgYnkgdGhpcyBuYXJyb3dlciBpbXBsZW1lbnRhdGlvbi4KCiMjIEh5cG90aGVzaXMgYW5kIGV4YWN0IG1lY2hhbmlzbQoKRm9yIHN1cHBvcnRlZCB0cmFpbmluZyBjYWxscywgcmV0YWluIHRoZSAqYWN0dWFsIGFscmVhZHktcm91bmRlZCBuYXRpdmUgRlAxNgpoZWFkIGxvZ2l0cyogYW5kIGNhdXNhbCB0YXJnZXRzIGluc3RlYWQgb2YgdGhlIG5hdGl2ZSBDRSBncmFwaCdzIEZQMzIKbG9nLXByb2JhYmlsaXRpZXMuIENvbXB1dGUgdGhlIHNjYWxhciB3aXRoIG5hdGl2ZSBmdWxsLWdlb21ldHJ5IHN1bW1lZCBDRSBpbgpmb3J3YXJkOyBsZWF2ZSB0aGUgb3JpZ2luYWwgc3VwcGxpZWQgZGl2aXNpb24gb3V0c2lkZSB0aGUgY3VzdG9tIG5vZGUuIER1cmluZwpiYWNrd2FyZCwgcmVjb21wdXRlIHRoZSBzYW1lIG5hdGl2ZSBzdW1tZWQgQ0UgZnJvbSB0aGUgc2F2ZWQgaGFsZiBsb2dpdHMgYW5kCmZlZWQgaXRzIGFjdHVhbCBpbmNvbWluZyAoYWxyZWFkeSBub3JtYWxpemVkKSB1cHN0cmVhbSBzY2FsYXIgaW50byBuYXRpdmUKYXV0b2dyYWQgYmVmb3JlIEZQMTYgZ3JhZGllbnQgcm91bmRpbmcuIFRoZXJlIGlzIG5vIGN1c3RvbSBzb2Z0bWF4LCBhbmFseXRpYyBkWiwKY2h1bmtlZCBDRSBzdW0sIHNlbGVjdGVkLXJvdyByZXBsYXksIGNoYW5nZWQgR0VNTSwgb3IgbW9kZWwvaGVhZCByZXBsYXkuCgpFeHBlY3RlZCBiZW5lZml0IGlzICoqbGVzcyByZXRhaW5lZCBsb3NzLWdyYXBoIHN0YXRlIGJlZm9yZSBiYWNrd2FyZCoqLiBOYXRpdmUKcmVwbGF5IGFkZHMgcmVhbCB3b3JrIGFuZCBjYW4gaW5jcmVhc2UgdGhlIGJhY2t3YXJkIHBlYWsuIFJldHVybmVkIEZQMzIgbG9naXRzCmFuZCBuYXRpdmUgU0ZUIGVudHJvcHkvYWNjdXJhY3kvUmV3aW5kIHJlbWFpbiBwcmVzZW50LiBUaGlzIGlzIG5vdCB0aGUgbGFyZ2UKbWVtb3J5LXNhdmluZyBIMCBjbGFpbSBhbmQgbm90IGEgcHJvbWlzZWQgY29tcGxldGUtcnVuIGdhaW4uCgojIyBTZW1hbnRpYyBib3VuZGFyaWVzCgotIFJ1bnRpbWUgYGFwcGx5KClgIGNoZWNrcyBUb3JjaCAyLjExLjArY3UxMjggYW5kIFRyYW5zZm9ybWVycyA1LjE2LjEsIGluc3RhbGxzCiAgb25seSBvbiB0aGUgZXhhY3QgbmF0aXZlIFF3ZW4yIGNvbnN0cnVjdG9yL3Byb3BlcnR5LCBhbmQgYXR0YWNoZXMgYSByZXZlcnNpYmxlCiAgcGVyLWluc3RhbmNlIGxvc3Mgd3JhcHBlci4gTm8gZ2xvYmFsIGxvc3MgbWFwLCBmb3J3YXJkLCBUcmFpbmVyLCBvcHRpbWl6ZXIsCiAgdGFzaywgc2FtcGxlciwgTG9SQSwgZHJvcG91dCwgc2NhbGVyLCBvdXRwdXQgb2JqZWN0LCBvciBSTkcgaXMgcGF0Y2hlZC4KLSBTdXBwb3J0ZWQgcGF0aDogYWN0dWFsIGNvbnRpZ3VvdXMgRlAxNiBgW0IsVCwxNTE5MzZdYCBsb2dpdHMsIG1hdGNoaW5nIHNhbWUtCiAgZGV2aWNlIGludDY0IGxhYmVscywgYSBmcm96ZW4gdW5iaWFzZWQgaGVhZCwgQ1VEQSBTTTc1LCBncmFkLWVuYWJsZWQgdHJhaW5pbmcsCiAgaGFyZCBjYXVzYWwgbGFiZWxzIHdpdGggaWdub3JlIGluZGV4IC0xMDAsIGFuZCBhbiBpbnRlZ2VyIG9yIHNjYWxhciBpbnQ2NAogIGRlbm9taW5hdG9yLiBBIENQVSBzY2FsYXIgZGVub21pbmF0b3IgZm9sbG93cyB0aGUgbmF0aXZlIGAudG8obG9zcy5kZXZpY2UpYC4KLSBPcmRpbmFyeSBgb3V0cHV0X2F0dGVudGlvbnNgL2BvdXRwdXRfaGlkZGVuX3N0YXRlc2AgYm9vbC9Ob25lIGt3YXJncyBhcmUKICBhY2NlcHRlZCBiZWNhdXNlIG5hdGl2ZSBRd2VuIGFscmVhZHkgY29uc3VtZWQgdGhlbSBhbmQgdGhlIHBpbm5lZCBsb3NzIGlnbm9yZXMKICB0aGVtLiBBbGwgb3RoZXIgZXh0cmEgYXJndW1lbnRzLCBhbHRlcm5hdGUgdGFyZ2V0cy9sYXlvdXRzL3ByZWNpc2lvbiwgZXZhbCwKICBtZWFuIHJlZHVjdGlvbiwgdHJhaW5hYmxlIGhlYWQgYW5kIHVuc3VwcG9ydGVkIHJvdXRlcyBjYWxsIHRoZSBjYXB0dXJlZCBuYXRpdmUKICBsb3NzICoqb24gdGhlIHNhbWUgbG9naXRzKiosIG5ldmVyIGJ5IHJldHJ5aW5nIGZvcndhcmQgYWZ0ZXIgUk5HIGNvbnN1bXB0aW9uLgotIEZ1bGwgQipUIGdlb21ldHJ5IHJlbWFpbnMsIGluY2x1ZGluZyB0aGUgZmluYWwgaWdub3JlZCByb3cgb2YgZWFjaCBleGFtcGxlLgogIElnbm9yZWQgTmFOL0luZiByb3dzLCB6ZXJvL25lZ2F0aXZlIGRlbm9taW5hdG9ycywgbm9uZmluaXRlIHVwc3RyZWFtIGdyYWRpZW50cywKICBhbmQgY29ubmVjdGVkLXplcm8gZ3JhZGllbnRzIGFyZSBub3Qgc2FuaXRpemVkLiBOYXRpdmUgRlAzMiBDRSBkZWZpbmVzIHRoZW0uCi0gU2F2ZWQgbG9naXRzIG11c3QgcmVtYWluIGltbXV0YWJsZSB1bnRpbCBiYWNrd2FyZC4gUHlUb3JjaCB2ZXJzaW9uIGNoZWNrcyBhcmUKICByZXRhaW5lZDsgbXV0YXRpb24gcGFyaXR5IG91dHNpZGUgdGhhdCBjb250cmFjdCBpcyBub3QgY2xhaW1lZC4gSGlnaGVyLW9yZGVyCiAgcmVwbGF5IGV4aXN0cyBidXQgaXMgdW52YWxpZGF0ZWQgYW5kIG5vdCBwYXJ0IG9mIHRoZSBmaXhlZCBmaXJzdC1vcmRlciB0YXNrLgotIGBTT1VQX0hFQURfTkFUSVZFYCBtZXRhZGF0YSBpcyBmbHVzaGVkIGJlZm9yZS9hZnRlciBldmVyeSBmb3J3YXJkL2JhY2t3YXJkLAogIGluY2x1ZGluZyBmYWxsYmFjayByZWFzb25zLiBJdCBzdG9yZXMgbm8gYWN0aXZhdGlvbnMuIFRoZXNlIGxpbmVzIHN1cnZpdmUgYQogIHByb2Nlc3MgdGltZW91dDsgdGhlaXIgb3ZlcmhlYWQgY291bnRzLiBUaGUgdW5jaGFuZ2VkIG9mZmljaWFsIGhhcm5lc3Mgc3RpbGwKICBvd25zIGFsbCBUcml0b24gbGF1bmNoL3NoYXJlIG1lYXN1cmVtZW50LiBOYXRpdmUgcmVwbGF5IGlzIG5ldmVyIGNhbGxlZCBhbgogIG93bi1rZXJuZWwgbGF1bmNoLgoKIyMgV2h5IGZ1bGwgSDAgYW5kIHN1bW1hcnktb25seSBmaW5pc2ggYXJlIGJsb2NrZWQKClJldmlld2VkIGxvY2FsIHJlY292ZXJlZCBSb3VuZCA3IHNvdXJjZXMsIGVzcGVjaWFsbHkgYHJvdW5kNl9pbnRlZ3JhdGlvbi9oMF9jb3JlLnB5YCwKYHJvdW5kN19sb3NzX2ludGVncmF0aW9uL3tSRUFETUUubWQsZm9yd2FyZF9zdGF0c19iYWNrZW5kLnB5LHN1bW1hcnlfc3RhdHMucHl9YCwKYW5kIGByb3VuZDdfY29tcG9zaXRpb24vYXV0b2Nhc3RfYm91bmRhcnlfcmVzdWx0cy5qc29uYCwgcGx1cyB0aGUgZnJlc2gKYHNvdXBfYXVkaXRfMTkxOS97aGFuZG9mZixsb3JhLHF3ZW59Lmpzb25gIGF1ZGl0cy4gVGhlc2UgYXJlIHByb3ZlbmFuY2UvZXZpZGVuY2UKaW5wdXRzLCAqKm5vdCBrZXJuZWxzIGNvcGllZCBpbnRvIHRoaXMgc29sdXRpb24gb3IgY3JlZGl0ZWQgYXMgb3VycyoqLgoKMS4gT21pdHRpbmcgb3V0cHV0IGxvZ2l0cyByZXF1aXJlcyBhbiBvd25lZCBTRlQvUmV3aW5kIHN1bW1hcnktY29uc3VtcHRpb24KICAgcHJvdG9jb2wsIHNpbmdsZS1mb3J3YXJkIHRpY2tldCwgdW5jaGFuZ2VkIHJvdyBpZGVudGl0aWVzL2NvdW50cyBhbmQgbGlmZWN5Y2xlCiAgIHRocm91Z2ggdGhlIG91dGVyIFJld2luZCByZWNvcmRlci4gQSBzYWZlIGFjdHVhbCBDVURBL1BFRlQvTkY0IGFkYXB0ZXIgaXMgbm90CiAgIGVzdGFibGlzaGVkIGJ5IHRoaXMgcHJlcGFyYXRpb24uIEFzc2lnbmluZyBgb3V0cHV0cy5sb2dpdHMgPSBOb25lYCBhbG9uZSBjYW4KICAgbGVhdmUgYSBkZW5zZSB0ZW5zb3IgaW4gdGhlIE1vZGVsT3V0cHV0IG1hcHBpbmc7IGEgcmVhbCByZWNvbnN0cnVjdGlvbiBhbmQKICAgY29tcGxldGUgY29uc3VtZXJzIHdvdWxkIGJlIHJlcXVpcmVkLiBUaGlzIGNhbmRpZGF0ZSBkb2VzIG5laXRoZXIgb21pc3Npb24uCjIuIEFjY2VsZXJhdGUgY2FuIGVudGVyIGF1dG9jYXN0ICppbnNpZGUqIHByZXBhcmVkIG1vZGVsIGZvcndhcmQgd2hpbGUgdHJhaW5lcgogICBlbnRyeSBpcyBvdXRzaWRlIGl0LiBUaGUgcmVjb3ZlcmVkIENQVSBCRjE2IHByb2JlIGRlbW9uc3RyYXRlcyB0aGlzIGJvdW5kYXJ5OwogICBtZXJlbHkgY2hhbmdpbmcgYW4gb3V0ZXIgQ1BVIGF1dG9jYXN0IGd1YXJkIHRvIENVREEgaXMgdW5zYWZlLgozLiBTdXBwbGllZCBzdW1tYXJ5LW9ubHkgZmluaXNoIHJlbW92ZXMgZm91ciB1bnVzZWQgcGVyLXRva2VuIG91dHB1dHMgKDMyIEtpQgogICBhdCAyMDQ4IHBvc2l0aW9ucyksIG5vdCB0aGUgaGFsZi1sb2dpdHMgY2FjaGUgb3IgbmF0aXZlIHJlcGxheS4gSXRzIHR3byBzdHJpY3QKICAgdGllLWVudHJvcHkgZmFpbHVyZXMgcmVtYWluOiBhYnNvbHV0ZSBlcnJvcnMgMi4yODg4MTgzNTkzNzVlLTUgKHRpbGUgMTAyNCkgYW5kCiAgIDIuMTkzNDUwOTI3NzM0Mzc1ZS01ICh0aWxlIDQwOTYpLiBTdXBwbGllZCBWMzcgbG9vcCBhZ3JlZW1lbnQsIGludGVycHJldGVyIGFuZAogICBTTTc1IG9mZmxpbmUgY29tcGlsYXRpb24gZG8gbm90IGVzdGFibGlzaCBhY3R1YWwgZnVsbC10YXNrIENVREEgYXJpdGhtZXRpYywKICAgc2hhcmUsIHNwZWVkIG9yIHBlYWsuIE5vIHN1cHBsaWVkIGNsYWltcyBhcmUgcmVsYWJlbGVkIGFzIG91ciBuZXcgdmFsaWRhdGlvbi4KClRoaXMgbG9zcy1ib3VuZGFyeSBjb250cm9sIGlzIHRoZXJlZm9yZSBkaXN0aW5jdCBmcm9tIGJvdGggdGhlIGZ1bGwgSDAgaHlicmlkCmFuZCB0aGUgc2VwYXJhdGUgbWF0ZXJpYWxpemVkL2FuYWx5dGljLUNFIGxhbmUuIEEgZnV0dXJlIG9yaWdpbmFsIHN1bW1hcnkgYnJpZGdlCndvdWxkIG5lZWQgaXRzIG93biBpbXBsZW1lbnRhdGlvbiwgZnJvemVuIG51bWVyaWNhbCBnYXRlcyBhbmQgdGFyZ2V0IGV2aWRlbmNlLgpUaGUgYWxnb3JpdGhtIGFsc28gZm9sbG93cyBvdXIgZWFybGllciBsb2NhbApgc291cF9jeWNsZS9wcml2YXRlX2hhbGZfbmF0aXZlX2NvbnRyb2wvY29udHJvbC5weWAgcmVzZWFyY2ggY29udHJvbDsgdGhpcyBwYWNrZXQKYWRkcyBhIHRhc2sgY29uc3RydWN0b3IgYWRhcHRlciwgZWFnZXIgdGVsZW1ldHJ5IGFuZCBhIG5ld2x5IGV4ZWN1dGVkIDI0LWNhc2UKc3VpdGUuIFRoZSBlYXJsaWVyIGluZGVwZW5kZW50IDIyLWNhc2UgY29udHJvbCByZXN1bHRzIHJlbWFpbiBzZXBhcmF0ZSBhbmQgd2VyZQpub3Qgb3ZlcndyaXR0ZW4gb3IgcmVsYWJlbGVkIGFzIHRoaXMgcGFja2V0J3MgcmVzdWx0cy4KCiMjIEZpbGVzIGFuZCBydW5uaW5nIGxhdGVyCgp8IEZpbGUgfCBQdXJwb3NlIHwKfC0tLXwtLS18CnwgYGNhbmRpZGF0ZS5pcHluYmAgfCBDbGVhcmVkLCBzZWxmLWNvbnRhaW5lZCBub3RlYm9vayB3aXRoIGV4YWN0IFpJUCArIGRyaXZlciBieXRlcyB8CnwgYHNvbHV0aW9uL2AgfCBBY3R1YWwgcmV2aWV3ZWQgc291cmNlOyBvbmx5IHRoaXMgZGlyZWN0b3J5IGlzIGp1ZGdlZC9leGVjdXRlZCB8CnwgYGNhbmRpZGF0ZS56aXBgLCBgY2FuZGlkYXRlLWZpbGVzLmpzb25gLCBgYnVuZGxlLWluZm8uanNvbmAgfCBEZXRlcm1pbmlzdGljIHdvcmtlciBidW5kbGUgYW5kIGV4YWN0IGhhc2hlcyB8CnwgYHJ1bl9jYW5kaWRhdGUucHlgIHwgT2ZmbGluZSBmcmVzaC1zdGFnZSBwYWlyZWQgb2ZmaWNpYWwtaGFybmVzcyBkcml2ZXIgfAp8IGBvZmZpY2lhbC1zb3VyY2UtbWFuaWZlc3QuanNvbmAgfCBQaW5uZWQgdW5jaGFuZ2VkIGhhcm5lc3MvdGFzay90ZXN0cyArIG90aGVyIG5vbi1zb2x1dGlvbiBzb3VyY2UgfAp8IGBzdWJtaXNzaW9uLnRlbXBsYXRlLmpzb25gIHwgUXVldWUtY29tcGF0aWJsZSB0ZW1wbGF0ZTsgYmFzZWxpbmUgSUQgaW50ZW50aW9uYWxseSB1bnJlc29sdmVkIHwKfCBgdmFsaWRhdGlvbi9gIHwgU291cmNlLWJvdW5kIGZvY3VzZWQgQ1BVL3VuaXQvc3RhdGljIHJlc3VsdHMsIG5vdCBHUFUgbWVhc3VyZW1lbnRzIHwKClRoZSBub3RlYm9vaydzIGZpcnN0IGNvbmZpZ3VyYXRpb24gY2VsbCByZXF1aXJlcyBhbiBhbHJlYWR5IHByZXBhcmVkIG9mZmljaWFsCmNoZWNrb3V0LCBleGlzdGluZyBkZWRpY2F0ZWQgcHVibGljIG1vZGVsIGNhY2hlLCBhIG5ldyBvdXRwdXQgcGF0aCwgYW5kIGV4cGxpY2l0CkdQVS1ydW4gYXV0aG9yaXphdGlvbi4gRGVmYXVsdCBHUFUgZXhlY3V0aW9uIGlzIGRpc2FibGVkLiBObyBpbnN0YWxsYXRpb24sCmRvd25sb2FkLCByZW1vdGUgZWRpdCwgcXVldWUgc3VibWlzc2lvbiwgb3IgR1BVIHJ1biBoYXBwZW5lZCBkdXJpbmcgcHJlcGFyYXRpb24uCgpBZnRlciBzZXBhcmF0ZSBleGVjdXRpb24gYXBwcm92YWwsIGNvbmZpZ3VyZSB0aGUgbm90ZWJvb2sgYW5kIFJ1biBBbGwsIG9yIHVzZToKCmBgYGJhc2gKcHl0aG9uIHJ1bl9jYW5kaWRhdGUucHkgXAogIC0tcmVwbyAvYWJzb2x1dGUvcHJlcGFyZWQvcGlubmVkLW9mZmljaWFsLWNoZWNrb3V0IFwKICAtLWNhY2hlIC9hYnNvbHV0ZS9wcmVwYXJlZC9wdWJsaWMtaGYtaHViLWNhY2hlIFwKICAtLW91dCAvYWJzb2x1dGUvbmV3L2hlYWQtZXhwZXJpbWVudC0wMSAtLWJhY2tlbmQgR0NQIC0tZXhlY3V0ZQpgYGAKClJlcXVpcmVzIFB5dGhvbiAzLjEzLjE1IGFuZCB0aGUgZXhhY3QgZGlyZWN0IHBhY2thZ2UgcGlucyByZWNvcmRlZCBpbiB0aGUgcXVldWUncwpgcmVxdWlyZW1lbnRzLWthZ2dsZS50eHRgOiBzb3VwLWNsaSAwLjc3LjAsIFRvcmNoIDIuMTEuMCtjdTEyOCwgVHJpdG9uIDMuNi4wLApUcmFuc2Zvcm1lcnMgNS4xNi4xLCBQRUZUIDAuMjAuMCwgVFJMIDAuMjkuMSwgYml0c2FuZGJ5dGVzIDAuNTAuMiwKQWNjZWxlcmF0ZSAxLjE0LjAgYW5kIGRhdGFzZXRzIDQuOC41LiBUaGlzIGlzIG5vdCBhIGZ1bGwgdHJhbnNpdGl2ZSBmcmVlemUuClRoZSBkcml2ZXIgdmVyaWZpZXMgb25lIHZpc2libGUgVGVzbGEgVDQsIFNNNzUvQ1VEQTEyLjgsIHN1bW1lZCBwaHlzaWNhbCBpZGxlCm1lbW9yeSA8MTAwMCBNaUIsIHB1YmxpYyBkYXRhIGhhc2hlcywgYW5kIG1vZGVsIHJldmlzaW9uCmA5ODlhYTc5ODBlNGNmODA2ZjgwYzdmZWYyYjFhZGI3YmM3MWFhMzA2YCBiZWZvcmUgdHJhaW5pbmcuCgpJdCBjcmVhdGVzIGEgbmV3IHN0YWdpbmcgcmVwb3NpdG9yeSBmcm9tIGhhc2gtY2hlY2tlZCBvZmZpY2lhbCBjb21taXQKYGM3YzJmY2QxNTk3MmNlOGI4OThkOGQzZmY3MzJlZTE5MmM4MTdjZWJgLCByZXBsYWNpbmcgb25seSBgc29sdXRpb24vYC4KSXQgcmV1c2VzIHB1YmxpYyBtb2RlbC9kYXRhIGFzc2V0cyBvZmZsaW5lIGFuZCBpbnZva2VzIHVuY2hhbmdlZApgaGFybmVzcy9ydW5fdGFzay5weSAtLXJlcGVhdHMgMiAtLXNlZWQgMWAgaW4gYmFzZS9zb2x1dGlvbi9zb2x1dGlvbi9iYXNlIG9yZGVyLgpFYWNoIG1lYXN1cmVkIHByb2Nlc3MgaGFzIGZyZXNoIG1vZGVsL3RyYWluaW5nIHN0YXRlLCBvdXRwdXRzLCBTb3VwIERCIGFuZApjb21waWxlciBjYWNoZXMuIFRoZSBuYXRpdmUgaGFybmVzcyBjYXB0dXJlcyBjb21wbGV0ZSBjb21tYW5kIHdhbGwgdGltZSwKZXh0ZXJuYWwgMTAwbXMgbnZpZGlhLXNtaSBwZWFrLCBwdWJsaWMgcXVhbGl0eSwgb3V0cHV0cyBhbmQgc3RlcHMzKyBrZXJuZWwgc2hhcmUuClRoZSBkcml2ZXIgYWRkaXRpb25hbGx5IHJlY29yZHMgd2hvbGUtb3JjaGVzdHJhdG9yIHdhbGwgdGltZSBzZXBhcmF0ZWx5LgpUaGUgY2Fub25pY2FsIG5vbmJsb2NraW5nIGAvdG1wL3NvdXAtZ3B1MC5sb2NrYCBjb3ZlcnMgdGhlIGVudGlyZSBwYWlyZWQgcnVuCmFuZCBjaGVja3MsIHByZXZlbnRpbmcgb3ZlcmxhcCB3aXRoIHRoZSBvdGhlciBwcmVwYXJlZCBub3RlYm9va3MuIEdQVSBwcmVmbGlnaHQKaXMgYSBzaG9ydC1saXZlZCBjaGlsZDsgaXQgZXhpdHMgYmVmb3JlIHRoZSBpZGxlIGNoZWNrLiBUaGUgZHJpdmVyL25vdGVib29rCmtlZXBzIG5vIGxpdmUgQ1VEQSBjb250ZXh0IGR1cmluZyBtZWFzdXJlbWVudC4KYC90bXAvc291cC1ncHUwLmxvY2suYmxvY2tlZGAgaXMgY2hlY2tlZCBiZWZvcmUgYW5kIGFmdGVyIGFjcXVpcmluZyB0aGUgbG9jay4KQW4gaXNvbGF0ZWQgTGludXggc3VicmVhcGVyIGd1YXJkaWFuIG93bnMgbmVzdGVkLXNlc3Npb24gY2hpbGRyZW4gYW5kIHNpZ25hbHMKb25seSBQSURGRC1ib3VuZCBkZXNjZW5kYW50czsgaXQgaW5oZXJpdHMgdGhlIGxvY2sgYW5kIHdhdGNoZXMgYSBwYXJlbnQtbGl2ZW5lc3MKcGlwZS4gVGltZW91dCwgaW50ZXJydXB0LCBwYXJlbnQgZGVhdGggb3IgdW5leHBlY3RlZCBkZXNjZW5kYW50cyBjcmVhdGVzIHRoZQpxdWFyYW50aW5lIG1hcmtlciBhbmQgY2xlYW5zL3JlYXBzIG93bmVkIGNoaWxkcmVuIGJlZm9yZSByZWxlYXNpbmcgdGhlIGxvY2suClJlcGVhdGVkIG5vdGVib29rIGludGVycnVwdHMgd2FpdCBmb3IgY2xlYW51cC4gTWFya2VycyBhcmUgbmV2ZXIgYXV0by1jbGVhcmVkOwphbiBvcGVyYXRvciBtdXN0IHJlY29uY2lsZSBiZWZvcmUgYW5vdGhlciBHUFUgcnVuLiBUaGlzIGlzIGJvdW5kZWQgdHJ1c3RlZC10YXNrCm9yY2hlc3RyYXRpb24sIG5vdCBhIGdlbmVyYWwgaG9zdGlsZS1jb2RlIHNhbmRib3guCgpJdCBydW5zIHVuY2hhbmdlZCBvZmZpY2lhbCBjaGVja3MgYWZ0ZXIgY29tcGxldGUgcGFpcmVkIHRyYWluL2V2YWwgZXZpZGVuY2UgYW5kCnByZXNlcnZlcyB0aGVpciBleHBlY3RlZCBvd24ta2VybmVsIGZhaWx1cmVzLiBGYWlsZWQvcGFydGlhbC90aW1lb3V0cyBnZXQgbm8KcXVhbGl0eSBjb21wYXJpc29uLCByYXRpb3MsIHNjb3JlcyBvciBhZG1pc3Npb24gUEFTUyBpbiBgcmVzZWFyY2gtcmVzdWx0Lmpzb25gLgpUaGUgdW50b3VjaGVkIHJhdyBvZmZpY2lhbCByZXN1bHQgcmVtYWlucyBldmlkZW5jZSwgbm90IGEgc2FmZXR5LWZpbHRlcmVkIHJlc3VsdC4KTm8gc2NvcmUgaXMgZXZlciBhc3NpZ25lZCBieSB0aGlzIGRyaXZlciwgZXZlbiBhZnRlciBhIGNvbXBsZXRlZCBjb250cm9sLgoKKipHQ1AgdXNlcyBpdHMgb3duIGZyZXNobHkgbWVhc3VyZWQgYmFzZWxpbmUsIG5ldmVyIHRoZSBoaXN0b3JpY2FsIEthZ2dsZSB+NzkgcwpsaW1pdC4qKiBUaGUgZGVmYXVsdCBuYXRpdmUgYnVkZ2V0IGlzIDkwMCBzIG9uIGJvdGggc2lkZXMsIGZpeGVkIGJlZm9yZSBydW5uaW5nOwp0aGUgd2hvbGUgcGFpcmVkLWhhcm5lc3Mgd2F0Y2hkb2cgaXMgODEwMCBzLiBHQ1AgZXZpZGVuY2UgaXMgbm90IG9mZmljaWFsIEthZ2dsZQphZG1pc3Npb24uIFRoZSB3b3JrZXIgYWNjZXB0cyB0aGUgWklQLCBub3QgZXhlY3V0aW9uIG9mIG5vdGVib29rIGNlbGxzOyBsZWF2ZQp0aGUgcXVldWUgdGVtcGxhdGUgdW50b3VjaGVkIHVudGlsIGEgcmV2aWV3ZXIgcHJvdmlkZXMgdGhlIGFjdHVhbCByZWdpc3RlcmVkCnNhbWUtZW52aXJvbm1lbnQgYmFzZWxpbmUgSUQgYW5kIHNlcGFyYXRlbHkgYXV0aG9yaXplcyBhIHJlc2VhcmNoIHJ1bi4KCiMjIFZhbGlkYXRpb24gY29tcGxldGVkIGhlcmUKCmB2YWxpZGF0aW9uL2NwdV9yZXN1bHRzLmpzb25gOiAqKjI0LzI0IENQVSBjaGVja3MgcGFzc2VkKiosIFRvcmNoIDIuMTUuMCtjcHUsClB5dGhvbiAzLjEyLjE0LCBUcmFuc2Zvcm1lcnMgNS4xNi4xLiBGaW5pdGUgbG9zcy9GUDE2LWdyYWRpZW50IHZhbHVlcyBhbmQgc2lnbmVkCnplcm9zIG1hdGNoZWQgZXhhY3RseSBpbiB0ZXN0ZWQgY2FzZXMsIHdpdGggbWF0Y2hpbmcgTmFOLytJbmYvLUluZiBwb3NpdGlvbnMuCkNhc2VzIGluY2x1ZGUgZnVsbCBWMTUxOTM2IGF0IEI0L1QzOyBUNTEyIGF0IFYxOTsgVDEvYWxsLWlnbm9yZWQ7IGxhcmdlL25vbmZpbml0ZQppbmNvbWluZyBzY2FsZTsgemVyby9uZWdhdGl2ZS90ZW5zb3IgZGVub21pbmF0b3I7IG5hdGl2ZSBmYWxsYmFjay9lcnJvciBpZGVudGl0eTsKc2F2ZWQtc3RhdGUgaW52ZW50b3J5IGFuZCBkZW5vbWluYXRvciB2ZXJzaW9uIGNoZWNrcy4KCkEgKip0aW55IHJlYWwgUXdlbjIgQ1BVKiogZml4dHVyZSBhZGRpdGlvbmFsbHkga2VlcHMgbmF0aXZlIG91dHB1dCBtYXBwaW5nLApoZWFkIG91dHB1dCwgbG9zcywgaW5wdXQgZ3JhZGllbnQsIGRyb3BvdXQgUk5HIGFuZCBzdGF0ZS1kaWN0IHN0cnVjdHVyZSBlcXVhbCwKdXNlcyBleGFjdGx5IG9uZSBuYXRpdmUgaGVhZCBmb3J3YXJkIHBlciBzaWRlIGFuZCB2ZXJpZmllcyBldmFsIGZhbGxiYWNrIHBsdXMKcmVzdG9yZS4gSXQgaXMgbm90IHRoZSBhY3R1YWwgcHJlcGFyZWQgUEVGVC9BY2NlbGVyYXRlL05GNC9UUkwvUmV3aW5kIHN0YWNrLgoKU2F2ZWQtc3RvcmFnZSBpbnZlbnRvcnkgaXMgc3RyaWN0bHkgYSBsb2NhbCByZXRhaW5lZC1hdXRvZ3JhZC1zdGF0ZSBvYnNlcnZhdGlvbiwKbm90IHByb2Nlc3MgcGVhay4gTWlzc2luZyB0YXJnZXQgcHJvb2ZzOiBhY3R1YWwgQ1VEQSBuYXRpdmUtbG9zcyBkaXNwYXRjaCwKVDQgbnVtZXJpY2FsL0FNUC9HcmFkU2NhbGVyIGJlaGF2aW9yLCByZWFsIFBFRlQvVFJML1Jld2luZCBpbnRlZ3JhdGlvbiwgY29tcGxldGUKNjQtc3RlcCBwYWlyZWQgcnVucywgcXVhbGl0eSwgZXh0ZXJuYWwgcGVhaywgd2hvbGUtY29tbWFuZCB0aW1lIGFuZCBmaW5hbCBqdXJ5CnJldmlldy4gT3duIFRyaXRvbiBwYXJ0aWNpcGF0aW9uIGlzIHN0cnVjdHVyYWxseSB6ZXJvLCBub3QgbWVyZWx5IHVubWVhc3VyZWQuCgpObyBHUFUgb3IgcmVtb3RlIHNlcnZpY2Ugd2FzIHRvdWNoZWQuIE5vIG9sZGVyIGZhaWxlZCBwcm90b3R5cGUgd2FzIG92ZXJ3cml0dGVuLgoKYHZhbGlkYXRpb24vcHJvY2Vzc19ndWFyZF9yZXN1bHRzLmpzb25gOiAqKjcvNyBDUFUgcHJvY2VzcyBmaXh0dXJlcyBwYXNzZWQqKjoKbmVzdGVkLXNlc3Npb24gdGltZW91dCwgU0lHSU5ULCBTSUdURVJNLCBhcmJpdHJhcnkgQmFzZUV4Y2VwdGlvbiwgYWJydXB0IHBhcmVudApTSUdLSUxMLCBvcnBoYW4gcmVwYXJlbnRpbmcsIGFuZCBub3JtYWwgY29tcGxldGlvbi4gRmFpbHVyZSBmaXh0dXJlcyB2ZXJpZnkgdGhhdAp0aGUgbmVzdGVkIENQVSBjaGlsZCBpcyBnb25lLCBxdWFyYW50aW5lIHJlbWFpbnMsIGFuZCB0aGUgaW5oZXJpdGVkIGxvY2sgaXMKcmVsZWFzZWQgb25seSBhZnRlciBjbGVhbnVwLiBGaXh0dXJlcyB1c2UgdGVtcG9yYXJ5IGxvY2tzL21hcmtlcnMsIG5vdCBHUFUgd29yawpvciB0aGUgY2Fub25pY2FsIHF1YXJhbnRpbmUgZmlsZS4gVGhlIHNoYXJlZCBQSURGRC9zdWJyZWFwZXIgaGVscGVyIGlzIGF0dHJpYnV0ZWQKdG8gdGhlIGNvbnRlbXBvcmFuZW91cyBMb1JBIGxhbmU7IGl0IGlzIG5vdCBzb2x1dGlvbiBjb2RlIG9yIGtlcm5lbCBjcmVkaXQuCg==', 'submission.template.json': 'ewogICJjYW5kaWRhdGVfaWQiOiAiaGVhZC1wcml2YXRlLWhhbGYtbmF0aXZlLXJlcGxheS0yMDI2MTAzMCIsCiAgImJhY2tlbmQiOiAiR0NQIiwKICAiYmFzZWxpbmVfaWQiOiAiUkVQTEFDRV9XSVRIX01FQVNVUkVEX0dDUF9CQVNFTElORV9JRCIsCiAgImVudHJ5cG9pbnQiOiAic291cF9zY3JlZW5fdjEiLAogICJhcnRpZmFjdF9raW5kIjogInNvbHV0aW9uX2J1bmRsZSIsCiAgInByaW9yaXR5IjogNTAsCiAgImNvbnRhaW5zX3NlY3JldHMiOiBmYWxzZQp9Cg=='}


In [ ]:
import base64
import hashlib
import json
import tempfile
import zipfile

PACKET = Path(tempfile.mkdtemp(prefix="soup-head-reviewed-"))
for name, payload in EMBEDDED_FILES.items():
    (PACKET / name).write_bytes(base64.b64decode(payload))
binding = json.loads((PACKET / "bundle-info.json").read_text())
manifest = json.loads((PACKET / "candidate-files.json").read_text())
assert hashlib.sha256((PACKET / "candidate.zip").read_bytes()).hexdigest() == binding["source_sha256"]
with zipfile.ZipFile(PACKET / "candidate.zip") as z:
    assert set(z.namelist()) == set(manifest)
    for name, expected in manifest.items():
        assert hashlib.sha256(z.read(name)).hexdigest() == expected
print("Prepared source:", binding["source_sha256"])
print("Status: T4 UNTESTED; native-replay control; own Triton share = 0; admission BLOCKED")
print("Packet:", PACKET)


## Authorized fixed-task comparison

Executing the following cell requires explicit authorization and all preflight
guards. It stages a **new** repository, leaves the prepared checkout untouched,
and invokes the unchanged official harness. Every measured run gets fresh
training state, output and compiler caches. Model/data assets are reused offline.
Compilation and native-CE recomputation count in complete time. Interrupted,
timed-out, incomplete or failed training/evaluation never becomes a score/PASS.


In [ ]:
import signal
import subprocess

if not AUTHORIZE_GPU_EXECUTION:
    raise RuntimeError("Prepared only: GPU execution is disabled. Obtain run approval and configure paths first.")
command = [sys.executable, str(PACKET / "run_candidate.py"),
           "--repo", str(PREPARED_REPO), "--cache", str(PUBLIC_HF_CACHE),
           "--out", str(OUTPUT), "--backend", BACKEND,
           "--timeout", str(NATIVE_TIMEOUT_S), "--outer-timeout", str(OUTER_TIMEOUT_S),
           "--execute"]
driver = subprocess.Popen(command, start_new_session=True)
try:
    returncode = driver.wait()
except KeyboardInterrupt:
    # Ask the driver to clean up its owned nested process tree and persist a
    # partial report. Do not abandon an executing GPU child on notebook stop.
    driver.send_signal(signal.SIGTERM)
    while True:
        try:
            returncode = driver.wait()
            break
        except KeyboardInterrupt:
            print("Waiting for owned-process cleanup; GPU lock remains held.")
print("Driver exit:", returncode)
report_path = OUTPUT / "research-result.json"
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps(report, indent=2))
    assert report["score"] is None
    assert report["admission"] != "PASS"
else:
    print("No complete driver report. Inspect preflight/driver error; do not infer success.")


## Interpretation and queue handoff

Read `research-result.json` together with all raw `measured/` results, logs,
official checks and source hashes. Native-replay activation requires 64 backward
completions in each candidate run; absence is not an optimization success.
Saved-state reduction does not prove lower external peak: replay can increase
backward peak and runtime. This packet does not suppress output logits or replace
native entropy, accuracy, Rewind, LoRA, dropout or RNG.

The worker consumes the exact reviewed `candidate.zip`, **not notebook cells**.
`submission.template.json` deliberately leaves the measured GCP baseline ID
unresolved. Do not submit until the reviewer authorizes the specific research
control and supplies the real registered cohort. Full H0 summary-only finish is
blocked pending an original safe consumer bridge and target numerical evidence;
recovered foreign kernels are not copied or credited as original work.
